In [12]:
"""
Eksperimen: use_dog=False vs use_dog=True
Bandingkan visual + fitur pada kedua ROI dari kamera HP
"""

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.feature import hog

# ── Konfigurasi (sama persis dengan notebook) ──
ROI_SIZE   = 200
IMAGE_SIZE = 64
HOG_ORIENT = 9
HOG_PIXELS = 8
HOG_CELLS  = 2
SGF_ANGLES = np.deg2rad(np.arange(0, 360, 15))
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)
GABOR_KSIZE   = 31
GABOR_SIGMA   = 4.0
GABOR_LAMBDA  = 20.0
GABOR_GAMMA   = 0.5
GABOR_THETAS  = [0, np.pi/4, np.pi/2, 3*np.pi/4]


def normalize_illumination(img_gray):
    img_f   = img_gray.astype(np.float32)
    g_small = cv2.GaussianBlur(img_f, (0, 0), sigmaX=1.0)
    g_large = cv2.GaussianBlur(img_f, (0, 0), sigmaX=10.0)
    dog     = g_small - g_large
    return cv2.normalize(dog, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def enhance_gabor(img_gray, use_dog=False):
    if use_dog:
        img_gray = normalize_illumination(img_gray)
    responses = []
    for theta in GABOR_THETAS:
        kernel = cv2.getGaborKernel(
            ksize=(GABOR_KSIZE, GABOR_KSIZE), sigma=GABOR_SIGMA,
            theta=theta, lambd=GABOR_LAMBDA, gamma=GABOR_GAMMA,
            psi=0, ktype=cv2.CV_32F
        )
        resp = cv2.filter2D(img_gray.astype(np.float32), cv2.CV_32F, kernel)
        responses.append(np.abs(resp))
    gabor_max = np.max(responses, axis=0)
    gabor_max = cv2.normalize(gabor_max, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    return clahe.apply(gabor_max)


def extract_hog_sgf(img_gray):
    img_64 = cv2.resize(img_gray, (IMAGE_SIZE, IMAGE_SIZE))
    hog_feat = hog(img_64, orientations=HOG_ORIENT,
                   pixels_per_cell=(HOG_PIXELS, HOG_PIXELS),
                   cells_per_block=(HOG_CELLS, HOG_CELLS),
                   block_norm='L2', visualize=False)
    img_f = img_64.astype(np.float32)
    Ix = cv2.Sobel(img_f, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(img_f, cv2.CV_32F, 0, 1, ksize=3)
    sgf_feats = []
    for theta in SGF_ANGLES:
        FR = np.cos(theta) * Ix + np.sin(theta) * Iy
        sgf_feats.extend([np.mean(FR), np.std(FR)])
    sgf_feat = np.array(sgf_feats, dtype=np.float32)
    hog_norm = hog_feat / (np.linalg.norm(hog_feat) + 1e-8)
    sgf_norm = sgf_feat / (np.linalg.norm(sgf_feat) + 1e-8)
    combined = np.concatenate([hog_norm * 0.8, sgf_norm * 0.2])
    norm = np.linalg.norm(combined)
    if norm > 0:
        combined = combined / norm
    return combined


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def image_stats(img):
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
    return {
        'blur_score': round(lap_var, 1),
        'brightness': round(float(img.mean()), 1),
        'contrast':   round(float(img.std()), 1),
    }


# ── Load kedua gambar ──
paths = [
    r"D:\xampp\htdocs\palmprint-backend\palmprint-ml\debug_inputs\debug_input_140815_170264.jpg",
    r"D:\xampp\htdocs\palmprint-backend\palmprint-ml\debug_inputs\debug_input_140815_270260.jpg",
]
names = ['ROI #1 (terang)', 'ROI #2 (gelap)']

rois = []
import os

for p in paths:
    print("\nMembaca:", p)
    print("File ada:", os.path.exists(p))

    img = cv2.imread(p)

    if img is None:
        print("GAGAL dibaca!")
        continue

    print("Berhasil:", img.shape)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (ROI_SIZE, ROI_SIZE))
    rois.append(gray)

# ── Proses tiap gambar dengan dua mode ──
results = {}
for roi, name in zip(rois, names):
    dog_img   = normalize_illumination(roi)
    enh_false = enhance_gabor(roi, use_dog=False)
    enh_true  = enhance_gabor(roi, use_dog=True)
    feat_false = extract_hog_sgf(enh_false)
    feat_true  = extract_hog_sgf(enh_true)
    results[name] = {
        'roi':        roi,
        'dog':        dog_img,
        'enh_false':  enh_false,
        'enh_true':   enh_true,
        'feat_false': feat_false,
        'feat_true':  feat_true,
        'stats_raw':       image_stats(roi),
        'stats_enh_false': image_stats(enh_false),
        'stats_enh_true':  image_stats(enh_true),
    }

# ── Hitung cosine similarity antar gambar ──
sim_ff = cosine_sim(results[names[0]]['feat_false'], results[names[1]]['feat_false'])
sim_tt = cosine_sim(results[names[0]]['feat_true'],  results[names[1]]['feat_true'])
sim_ft = cosine_sim(results[names[0]]['feat_false'], results[names[1]]['feat_true'])

print("=" * 55)
print("  COSINE SIMILARITY antara ROI #1 dan ROI #2")
print("=" * 55)
print(f"  use_dog=False vs use_dog=False : {sim_ff:.4f}")
print(f"  use_dog=True  vs use_dog=True  : {sim_tt:.4f}")
print(f"  use_dog=False vs use_dog=True  : {sim_ft:.4f}")
print()
print("  Threshold sistem: 0.16")
print(f"  use_dog=False → {'MATCH ✓' if sim_ff >= 0.16 else 'TIDAK MATCH ✗'} (sim={sim_ff:.4f})")
print(f"  use_dog=True  → {'MATCH ✓' if sim_tt >= 0.16 else 'TIDAK MATCH ✗'} (sim={sim_tt:.4f})")
print("=" * 55)

print()
print("  STATS GAMBAR:")
for name in names:
    r = results[name]
    print(f"\n  {name}")
    print(f"    RAW         : blur={r['stats_raw']['blur_score']}, bright={r['stats_raw']['brightness']}, contrast={r['stats_raw']['contrast']}")
    print(f"    Gabor(False): blur={r['stats_enh_false']['blur_score']}, bright={r['stats_enh_false']['brightness']}, contrast={r['stats_enh_false']['contrast']}")
    print(f"    Gabor(True) : blur={r['stats_enh_true']['blur_score']}, bright={r['stats_enh_true']['brightness']}, contrast={r['stats_enh_true']['contrast']}")

# ── Visualisasi ──
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Eksperimen: use_dog=False vs use_dog=True pada ROI dari kamera HP',
             fontsize=13, fontweight='bold')

col_labels = ['1. ROI asli', '2. DoG saja', '3. Gabor (use_dog=False)',
              '4. Gabor+DoG (use_dog=True)', '5. Feature vector']

for row, name in enumerate(names):
    r = results[name]

    axes[row, 0].imshow(r['roi'], cmap='gray')
    axes[row, 0].set_title(f'{col_labels[0]}\n{name}', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(r['dog'], cmap='gray')
    axes[row, 1].set_title(f'{col_labels[1]}\nblur={r["stats_raw"]["blur_score"]}', fontsize=9, fontweight='bold')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(r['enh_false'], cmap='gray')
    s = r['stats_enh_false']
    axes[row, 2].set_title(f'{col_labels[2]}\nblur={s["blur_score"]} bright={s["brightness"]}', fontsize=9, fontweight='bold')
    axes[row, 2].axis('off')

    axes[row, 3].imshow(r['enh_true'], cmap='gray')
    s = r['stats_enh_true']
    axes[row, 3].set_title(f'{col_labels[3]}\nblur={s["blur_score"]} bright={s["brightness"]}', fontsize=9, fontweight='bold')
    axes[row, 3].axis('off')

    # Feature vector comparison
    hog_dim = len(r['feat_false']) - len(SGF_ANGLES) * 2
    axes[row, 4].plot(r['feat_false'], lw=0.7, color='steelblue', alpha=0.8, label='use_dog=False')
    axes[row, 4].plot(r['feat_true'],  lw=0.7, color='darkorange', alpha=0.8, label='use_dog=True')
    axes[row, 4].set_title(f'{col_labels[4]}\n1812d L2-norm', fontsize=9, fontweight='bold')
    axes[row, 4].legend(fontsize=7)
    axes[row, 4].grid(True, alpha=0.3)

# Tambah anotasi similarity di bagian bawah
fig.text(0.5, 0.01,
    f'Cosine similarity ROI#1 vs ROI#2 — '
    f'use_dog=False: {sim_ff:.4f}  |  use_dog=True: {sim_tt:.4f}  '
    f'(threshold: 0.16)',
    ha='center', fontsize=11, fontweight='bold',
    color='green' if sim_tt > sim_ff else 'navy')

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('dog_experiment.png', dpi=130, bbox_inches='tight')
print("\n  Visualisasi disimpan: dog_experiment.png")


Membaca: D:\xampp\htdocs\palmprint-backend\palmprint-ml\debug_inputs\debug_input_140815_170264.jpg
File ada: True
Berhasil: (200, 200, 3)

Membaca: D:\xampp\htdocs\palmprint-backend\palmprint-ml\debug_inputs\debug_input_140815_270260.jpg
File ada: True
Berhasil: (200, 200, 3)
  COSINE SIMILARITY antara ROI #1 dan ROI #2
  use_dog=False vs use_dog=False : 0.9249
  use_dog=True  vs use_dog=True  : 0.9262
  use_dog=False vs use_dog=True  : 0.7415

  Threshold sistem: 0.16
  use_dog=False → MATCH ✓ (sim=0.9249)
  use_dog=True  → MATCH ✓ (sim=0.9262)

  STATS GAMBAR:

  ROI #1 (terang)
    RAW         : blur=760.5, bright=166.7, contrast=28.8
    Gabor(False): blur=13.3, bright=140.1, contrast=50.4
    Gabor(True) : blur=58.2, bright=162.9, contrast=45.2

  ROI #2 (gelap)
    RAW         : blur=372.4, bright=173.2, contrast=29.4
    Gabor(False): blur=16.3, bright=156.0, contrast=49.7
    Gabor(True) : blur=74.1, bright=158.9, contrast=42.5

  Visualisasi disimpan: dog_experiment.png
